# Bayesian SARIMA + SGMCMC Streaming Anomaly Detection
**논문 구조와 일치: SARIMAX + 칼만 필터 + SGMCMC**

- MLE 베이스라인: `fit()` → `extend()` (기존 Code_Streaming 코드)
- SGMCMC (GD/SGD/SGLD/SGHMC/SGNHT): `fit()`으로 초기화 → `score()`로 gradient → 사전분포 포함 업데이트
- 성능 평가: 포인트 단위 accuracy/precision/recall/f1 → 배치 평균

## 1. Kafka & Zookeeper 시작

In [1]:
import os, time, subprocess

STREAMING_DIR = os.path.expanduser('~/Streaming')
os.chdir(STREAMING_DIR)
KAFKA_DIR = os.path.join(STREAMING_DIR, 'kafka_2.13-3.7.1')

!pkill -f kafka 2>/dev/null || true
!pkill -f zookeeper 2>/dev/null || true
time.sleep(3)
!rm -rf /tmp/zookeeper /tmp/kafka-logs

zk = subprocess.Popen(
    [f'{KAFKA_DIR}/bin/zookeeper-server-start.sh', f'{KAFKA_DIR}/config/zookeeper.properties'],
    stdout=open('/tmp/zk.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(10)

kafka = subprocess.Popen(
    [f'{KAFKA_DIR}/bin/kafka-server-start.sh', f'{KAFKA_DIR}/config/server.properties'],
    stdout=open('/tmp/kafka.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(15)

!ss -tlnp | grep -E '(2181|9092)'
print('\n✅ Kafka & Zookeeper 시작 완료')

LISTEN 0      50                      *:9092             *:*    users:(("java",pid=2150323,fd=156))         
LISTEN 0      50                      *:2181             *:*    users:(("java",pid=2149852,fd=144))         

✅ Kafka & Zookeeper 시작 완료


## 2. 라이브러리 임포트

In [2]:
import numpy as np
import pandas as pd
import pickle, math, time, json, requests, statistics, warnings

from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, confusion_matrix)
from sklearn import metrics
from kafka import KafkaProducer
import tensorflow as tf
import tensorflow_io as tfio

warnings.filterwarnings('ignore')
print('✅ 라이브러리 임포트 완료')

2026-03-03 09:07:22.994083: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-03 09:07:23.020971: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-03 09:07:23.020994: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-03 09:07:23.021892: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-03 09:07:23.026488: I tensorflow/core/platform/cpu_feature_guar

✅ 라이브러리 임포트 완료


## 3. 데이터 로드 & 전처리

In [38]:
# 데이터셋 선택
DATASET = 'realKnownCause/machine_temperature_system_failure.csv'
BATCH_SIZE = 48
ORDER = (0, 1, 1)
SEASONAL_ORDER = (1, 1, 0, 48)
SIGMA_LEVEL = 1.2  # 신뢰구간 배수

# 데이터 로드
static_data_url = f'https://raw.githubusercontent.com/numenta/NAB/master/data/{DATASET}'
static_data_df = pd.read_csv(static_data_url)

# 이상치 라벨
labels_url = 'https://raw.githubusercontent.com/numenta/NAB/master/labels/combined_labels.json'
anomaly_labels = json.loads(requests.get(labels_url).text)
anomaly_labels_df = pd.DataFrame(anomaly_labels[DATASET], columns=['timestamp'])
anomaly_labels_df['anomaly_label'] = True
static_data_labeled_df = pd.merge(static_data_df, anomaly_labels_df, on='timestamp', how='left')
static_data_labeled_df['anomaly_label'] = static_data_labeled_df['anomaly_label'].fillna(False).astype(int)
static_data_labeled_df = static_data_labeled_df.dropna()
static_data_labeled_df['timestamp'] = pd.to_datetime(static_data_labeled_df['timestamp'])
static_data_labeled_df['timestamp'] = static_data_labeled_df['timestamp'].astype('int64') // 10**9

# Train/Test 분할
train_df, test_df = train_test_split(static_data_labeled_df, test_size=0.2, shuffle=False)
x_train_df = train_df.drop(['anomaly_label'], axis=1)
y_train_df = train_df['anomaly_label']
x_test_df = test_df.drop(['anomaly_label'], axis=1)
y_test_df = test_df['anomaly_label']

x_train = list(filter(None, x_train_df.to_csv(index=False).split('\n')[1:]))
y_train = list(filter(None, y_train_df.to_csv(index=False).split('\n')[1:]))
x_test = list(filter(None, x_test_df.to_csv(index=False).split('\n')[1:]))
y_test = list(filter(None, y_test_df.to_csv(index=False).split('\n')[1:]))

batch_size = BATCH_SIZE  # 기존 코드 호환

print(f'📊 데이터셋: {DATASET}')
print(f'   Train: {len(train_df)}, Test: {len(test_df)}')
print(f'   이상치: {static_data_labeled_df["anomaly_label"].sum()}개')

📊 데이터셋: realKnownCause/machine_temperature_system_failure.csv
   Train: 18156, Test: 4539
   이상치: 4개


In [42]:
print(f'GLOBAL_MIN: {GLOBAL_MIN}, GLOBAL_MAX: {GLOBAL_MAX}')
# 데이터 간격 확인
static_data_labeled_df['ts'] = pd.to_datetime(static_data_labeled_df['timestamp'], unit='s')
print(f"데이터 간격: {static_data_labeled_df['ts'].diff().mode().values[0]}")

GLOBAL_MIN: 2.084721206, GLOBAL_MAX: 108.5105428
데이터 간격: 300000000000 nanoseconds


In [41]:
# Train/Test 분할 직후에 전역 최소/최대값 추출
GLOBAL_MIN = train_df['value'].min() # value 컬럼명에 맞게 수정
GLOBAL_MAX = train_df['value'].max()

## 4. Kafka 유틸리티

In [43]:
def error_callback(e):
    print(f'Kafka Error: {e}')

def write_to_kafka(topic_name, items):
    producer = KafkaProducer(bootstrap_servers=['127.0.0.1:9092'])
    count = 0
    for message, key in items:
        producer.send(topic_name, key=key.encode('utf-8'), 
                      value=message.encode('utf-8')).add_errback(error_callback)
        count += 1
    producer.flush()
    producer.close()
    print(f'✅ {topic_name}: {count}개 전송 완료')

def decode_kafka_online_item(raw_message, raw_key):
    message = tf.io.decode_csv(raw_message, record_defaults=[[0.0], [0.0]])
    key = tf.strings.to_number(raw_key)
    return (tf.stack(message, axis=0), key)

print('✅ Kafka 유틸리티 정의 완료')

✅ Kafka 유틸리티 정의 완료


## 5. 성능 평가 함수 (기존 코드와 동일)

In [44]:
def calculating_batch_confidence_bound(timestamp_column, predicted_values_column, sigma_level):
    values_column = predicted_values_column
    batch_mean = np.mean(values_column)
    batch_std = np.std(values_column)
    batch_mean_upper_bound = batch_mean + sigma_level * batch_std
    batch_mean_lower_bound = batch_mean - sigma_level * batch_std
    batch_data = [batch_std, batch_mean_lower_bound, batch_mean_upper_bound]
    return batch_data

In [45]:
def evaluate_forecast(loaded_model, minids_message, minids_key, 
                      current_message_count, batch_conf_interval,
                      perf_acc, perf_prec, perf_rec, perf_f1,
                      perf_mse, perf_rmse, perf_mae,
                      cnt_anomalies, actual_anomalies, tp_anomalies, method='mle'):
    global param_samples, current_params
    timestamp_column = minids_message[:, 0]
    values_column1 = minids_message[:, 1].astype(float)
    
    values_column = (values_column1 - GLOBAL_MIN) / (GLOBAL_MAX - GLOBAL_MIN + 1e-8)
    
    y_labels = minids_key
    forecast_steps = len(values_column)

    try:
        bma_uncertainty = None
        if current_params is not None:
            forecast = saved_model_result.get_forecast(steps=forecast_steps)
            predicted_values = forecast.predicted_mean
            
            # ✅ SGLD: 파라미터 샘플의 σ² 분산으로 불확실성 직접 계산
            if method in ['sgld', 'sgd', 'sghmc','sgnht'] and len(param_samples) > 1:
                sigma2_samples = np.array([p[2] for p in param_samples])
                bma_uncertainty = np.std(sigma2_samples) / np.mean(sigma2_samples)
                
                # method별 계수
                if method == 'sghmc':
                    coeff = 0.01
                elif method == 'sgnht':
                    coeff = 0.3   # thermostat 덕에 안정적이라 좀 더 크게
                elif method == 'sgld':
                    coeff = 0.5
                else:  # sgd
                    coeff = 0.5
                
                adaptive_interval = batch_conf_interval * (1 + coeff * bma_uncertainty)
            else:
                adaptive_interval = batch_conf_interval
        else:
            forecast = loaded_model.get_forecast(steps=forecast_steps)
            predicted_values = forecast.predicted_mean
    except:
        predicted_values = np.full(forecast_steps, np.mean(values_column))
        bma_uncertainty = None

    residuals = predicted_values - values_column

    # ---------- [🚨 맨 처음 원본 파일의 판별 로직 100% 복구!] ----------
    # Residual(오차)이 아닌 predicted_values(예측값)의 파동을 검사하던 원본의 방식입니다.
    batch_mean = np.mean(predicted_values)
    batch_std = np.std(predicted_values)
    if bma_uncertainty is not None and bma_uncertainty > 0:
        adaptive_interval = batch_conf_interval * (1 + 0.5 * bma_uncertainty)
    else:
        adaptive_interval = batch_conf_interval

    batch_data_lower = batch_mean - (adaptive_interval * batch_std) 
    batch_data_upper = batch_mean + (adaptive_interval * batch_std)
    
    anomalies_conf_batch = (predicted_values < batch_data_lower) | (predicted_values > batch_data_upper)
    # -------------------------------------------------------------------

    mse = np.mean(residuals ** 2)
    perf_mse[current_message_count] = mse
    perf_rmse[current_message_count] = np.sqrt(mse)
    perf_mae[current_message_count] = np.mean(np.abs(residuals))

    batch_detected = int(anomalies_conf_batch.any())
    batch_actual = int(y_labels.sum() > 0)

    if batch_detected:
        cnt_anomalies[current_message_count] = 1
    if batch_actual:
        actual_anomalies[current_message_count] = 1
    if batch_detected and batch_actual:
        tp_anomalies[current_message_count] = 1

    perf_acc[current_message_count] = int(batch_detected == batch_actual)
    perf_prec[current_message_count] = 1.0 if (batch_detected and batch_actual) else (0.0 if batch_detected else 1.0)
    perf_rec[current_message_count] = 1.0 if (batch_detected and batch_actual) else (0.0 if batch_actual else 1.0)
    perf_f1[current_message_count] = 1.0 if (batch_detected and batch_actual) else 0.0 if (batch_detected or batch_actual) else 1.0

## 6. 모델 학습 함수 (MLE vs SGMCMC 통합)

- **MLE**: 기존 `fit_sarimax_model`과 동일 → `fit()` + `extend()`
- **SGMCMC**: `fit()`으로 초기화 → 칼만필터 `score()`로 gradient → 사전분포 포함 업데이트

In [46]:
PRIOR_MEAN = 0.0
PRIOR_VAR = 10.0
GRAD_CLIP = 1.0

saved_model_result = None
data_history = []
saved_model_result = None
data_history = []
param_samples = []


def fit_model(minids_message, current_message_count, 
              method='mle', learning_rate=1e-5):
    global current_params, current_state, momentum, thermostat, data_history, saved_model_result, param_samples
    
    # 1. 배치 데이터 로드 및 전역 정규화(Global Normalization)
    value_column = minids_message[:, 1].astype(float)
    normalized = (value_column - GLOBAL_MIN) / (GLOBAL_MAX - GLOBAL_MIN + 1e-8)

    if method == 'mle':
        if current_message_count == 1:
            model = SARIMAX(normalized, order=ORDER, seasonal_order=SEASONAL_ORDER,
                            seasonal=True, enforce_invertibility=False)
            saved_model_result = model.fit(disp=False)
        else:
            saved_model_result = saved_model_result.extend(endog=normalized)
        return

    # 데이터 히스토리 누적
    data_history.extend(normalized.tolist())
    MAX_HISTORY = 48 * 2
    if len(data_history) > MAX_HISTORY:
        data_history = data_history[-MAX_HISTORY:]

    # 모델 초기화 (첫 배치)
    if current_params is None:
        model = SARIMAX(normalized, order=ORDER, seasonal_order=SEASONAL_ORDER,
                        seasonal=True, enforce_invertibility=False)
        res = model.fit(disp=False)
        current_params = res.params.copy()
        momentum = np.zeros_like(current_params)
        thermostat = 1.0
        saved_model_result = res
        return

    # MCMC 파라미터 업데이트
    UPDATE_INTERVAL = 5
    if current_message_count % UPDATE_INTERVAL == 0:
        cumulative_data = np.array(data_history)
        grad_model = SARIMAX(cumulative_data, order=ORDER, seasonal_order=SEASONAL_ORDER,
                             seasonal=True, enforce_invertibility=False)
        d = len(current_params)
        eps = learning_rate
        scale = len(x_train) / len(cumulative_data)

        # Gradient 계산 및 클리핑
        try:
            lik_grad = grad_model.score(current_params)
            g_norm = np.linalg.norm(lik_grad)
            if g_norm > GRAD_CLIP:
                lik_grad = lik_grad * GRAD_CLIP / g_norm
        except:
            lik_grad = np.zeros(d)

        prior_grad = -(current_params - PRIOR_MEAN) / PRIOR_VAR
        full_grad = scale * lik_grad + prior_grad

        # ==========================================
        # 🌟 5가지 방법론 분기 (GD, SGD, SGLD, SGHMC, SGNHT)
        # ==========================================
        if method == 'gd':
            current_params = current_params + eps * full_grad
            
        elif method == 'sgd':
            mini_batch_size = min(24, len(cumulative_data))
            idx = np.random.choice(len(cumulative_data), mini_batch_size, replace=False)
            sgd_data = cumulative_data[idx]
            
            sgd_model = SARIMAX(sgd_data, order=ORDER, seasonal_order=SEASONAL_ORDER,
                                seasonal=True, enforce_invertibility=False)
            try:
                sgd_grad = sgd_model.score(current_params)
                g_norm = np.linalg.norm(sgd_grad)
                if g_norm > GRAD_CLIP:
                    sgd_grad = sgd_grad * GRAD_CLIP / g_norm
            except:
                sgd_grad = np.zeros(d)
            
            prior_grad = -(current_params - PRIOR_MEAN) / PRIOR_VAR
            sgd_full_grad = scale * sgd_grad + prior_grad
            
            # 🌟 [수정 완료] 노이즈 없이 순수 미니배치 기울기로만 업데이트! (모멘텀 삭제)
            current_params = current_params + eps * sgd_full_grad 
            
            BURNIN = 20
            if current_message_count > BURNIN:
                param_samples.append(current_params.copy())
                if len(param_samples) > 50:
                    param_samples.pop(0)
            
        elif method == 'sgld':
            noise_scale = 1e-2  # 기존 sqrt(2*1e-5) ≈ 0.004 → 0.01로 증가
            noise = noise_scale * np.random.randn(d)
            current_params = current_params + eps * full_grad + noise
            
            BURNIN = 20
            if current_message_count > BURNIN:
                param_samples.append(current_params.copy())
                if len(param_samples) > 50:
                    param_samples.pop(0)
            
        elif method == 'sghmc':
            alpha = 50.0
            momentum = (1 - alpha * eps) * momentum + eps * full_grad
            m_norm = np.linalg.norm(momentum)
            if m_norm > 1.0: momentum = momentum / m_norm
            current_params = current_params + momentum
            # 파라미터에 직접 노이즈 (SGLD보다 작게)
            current_params = current_params + 5e-3 * np.random.randn(d)

            
        elif method == 'sgnht':
            A = 0.01
            target_temp = 0.001
            momentum = momentum + eps * full_grad - eps * thermostat * momentum
            
            m_norm = np.linalg.norm(momentum)
            if m_norm > 1.0: momentum = momentum / m_norm
                
            kinetic_energy = np.dot(momentum, momentum) / d
            thermostat = thermostat + eps * (kinetic_energy - target_temp)
            thermostat = np.clip(thermostat, 0.0, 10.0)
            current_params = current_params + momentum * eps
            # 파라미터에 직접 노이즈 (SGHMC와 SGLD 사이)
            current_params = current_params + 7e-3 * np.random.randn(d)

        current_params[0] = np.clip(current_params[0], -1.5, 1.5)
        current_params[1] = np.clip(current_params[1], -1.5, 1.5)
        current_params[2] = max(current_params[2], 0.0001)
        
        if method in ['sghmc', 'sgnht']:
            BURNIN = 20
            if current_message_count > BURNIN:
                param_samples.append(current_params.copy())
                if len(param_samples) > 50:
                    param_samples.pop(0)

    if current_params is not None:
        # ★ 핵심: SGLD 노이즈가 섞인 파라미터를 칼만 필터 예측에 강제로 꽂아 넣습니다!
        saved_model_result = saved_model_result.apply(normalized, params=current_params)
    else:
        saved_model_result = saved_model_result.extend(endog=normalized)

## 7. Kafka 토픽 생성 (1회)

In [47]:
TRAIN_TOPIC = f'train-{int(time.time())}'
TEST_TOPIC = f'test-{int(time.time())}'

os.system(f'{KAFKA_DIR}/bin/kafka-topics.sh --create --bootstrap-server 127.0.0.1:9092 '
          f'--replication-factor 1 --partitions 1 --topic {TRAIN_TOPIC} 2>/dev/null')
os.system(f'{KAFKA_DIR}/bin/kafka-topics.sh --create --bootstrap-server 127.0.0.1:9092 '
          f'--replication-factor 1 --partitions 1 --topic {TEST_TOPIC} 2>/dev/null')
time.sleep(1)

write_to_kafka(TRAIN_TOPIC, zip(x_train, y_train))
write_to_kafka(TEST_TOPIC, zip(x_test, y_test))
print(f'\n✅ 토픽 생성 완료: {TRAIN_TOPIC}, {TEST_TOPIC}')

Created topic train-1772499137.
Created topic test-1772499137.
✅ train-1772499137: 18156개 전송 완료
✅ test-1772499137: 4539개 전송 완료

✅ 토픽 생성 완료: train-1772499137, test-1772499137


In [48]:
def print_results(perf_acc, perf_prec, perf_rec, perf_f1, 
                  perf_mse, perf_rmse, perf_mae,
                  cnt_anomalies, actual_anomalies, tp_anomalies,
                  phase='TRAIN', method='mle', elapsed=0):
    
    total_batches = len(perf_acc)
    if total_batches == 0:
        print("⚠️ 데이터 없음 - Kafka 토픽을 확인하세요")
        return {'recall': 0, 'accuracy': 0, 'precision': 0, 'f1': 0, 'time': 0}

    total_alarms = len(cnt_anomalies)
    total_actual = len(actual_anomalies)
    total_tp = len(tp_anomalies)
    
    fp = total_alarms - total_tp
    fn = total_actual - total_tp
    tn = total_batches - total_alarms - fn
    
    accuracy = (total_tp + tn) / total_batches * 100 if total_batches > 0 else 0
    precision = total_tp / total_alarms * 100 if total_alarms > 0 else 0
    recall = total_tp / total_actual * 100 if total_actual > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    print(f'\n{"="*60}')
    print(f'📊 {phase} 결과 ({method.upper()})')
    print(f'{"="*60}')
    print(f'총 배치      : {total_batches}')
    print(f'알람 배치    : {total_alarms}')
    print(f'실제 이상 배치: {total_actual}')
    print(f'TP 배치      : {total_tp}')
    print(f'MSE Mean     : {statistics.mean(perf_mse.values()):.8f}')
    print(f'RMSE Mean    : {statistics.mean(perf_rmse.values()):.6f}')
    print(f'MAE Mean     : {statistics.mean(perf_mae.values()):.6f}')
    print(f'Accuracy     : {accuracy:.2f}%')
    print(f'Precision    : {precision:.2f}%')
    print(f'Recall       : {recall:.2f}%')
    print(f'F1 Score     : {f1:.2f}%')
    print(f'Time         : {elapsed:.1f}s')
    print(f'TP Batches   : {sorted(tp_anomalies.keys())}')
    print(f'{"="*60}')
    
    return {
        'recall': round(recall, 2),
        'accuracy': round(accuracy, 2),
        'precision': round(precision, 2),
        'f1': round(f1, 2),
        'time': round(elapsed, 1),
    }

## 8. Train 실행 함수

In [49]:
def run_train(method, learning_rate=1e-5, batch_conf_interval=1.6):
    global current_params, current_state, momentum, thermostat, data_history, saved_model_result, param_samples
     
    current_params = None
    current_state = None
    momentum = None
    thermostat = None
    data_history = []
    param_samples = []

    saved_model_result = None
    if os.path.exists('sarimax_model.pkl'):
        os.remove('sarimax_model.pkl')
    
    # 성능 지표 딕셔너리 (기존 코드와 동일 구조)
    perf_acc, perf_prec, perf_rec, perf_f1 = {}, {}, {}, {}
    perf_mse, perf_rmse, perf_mae = {}, {}, {}
    cnt_anomalies, actual_anomalies, tp_anomalies = {}, {}, {}
    
    # Kafka 데이터셋
    online_ds = tfio.experimental.streaming.KafkaBatchIODataset(
        topics=[TRAIN_TOPIC],
        group_id=f'cg_train_{method}_{int(time.time())}',
        servers='127.0.0.1:9092',
        stream_timeout=10000,
        configuration=['session.timeout.ms=7000',
                       'max.poll.interval.ms=8000',
                       'auto.offset.reset=earliest'],
    )
    
    iterations_count = math.ceil(len(x_train) / batch_size) - 1
    current_message_count = 0
    prev_message = None
    
    print(f'\n🚀 TRAIN 시작 - {method.upper()}, lr={learning_rate}')
    train_start = time.time()
    
    for mini_ds in online_ds:
        mini_ds = mini_ds.map(decode_kafka_online_item)
        mini_ds = mini_ds.batch(batch_size)
        
        for (message, key) in mini_ds:
            if current_message_count >= iterations_count:
                break
            current_message_count += 1
            
            msg_np = message.numpy()
            key_np = key.numpy()
            
            # 학습
            fit_model(msg_np, current_message_count, 
                      method=method, learning_rate=learning_rate)
            
            # 예측 + 평가
            evaluate_forecast(saved_model_result, msg_np, key_np, current_message_count,
                              batch_conf_interval,
                              perf_acc, perf_prec, perf_rec, perf_f1,
                              perf_mse, perf_rmse, perf_mae,
                              cnt_anomalies, actual_anomalies, tp_anomalies, method=method)
            
            if current_message_count % 50 == 0:
                print(f'  Batch {current_message_count}/{iterations_count}')
        
        if current_message_count >= iterations_count:
            break
    
    elapsed = time.time() - train_start
    if param_samples:
        arr = np.array(param_samples)
        print(f"param_samples std: {np.std(arr, axis=0)}")
    return print_results(perf_acc, perf_prec, perf_rec, perf_f1,
                         perf_mse, perf_rmse, perf_mae,
                         cnt_anomalies, actual_anomalies, tp_anomalies,
                         phase='TRAIN', method=method, elapsed=elapsed)


print('✅ Train 함수 정의 완료')

✅ Train 함수 정의 완료


## 9. Test 실행 함수

In [50]:
def run_test(method, learning_rate=1e-5, batch_conf_interval=1.6):
    # 🌟 1. param_samples를 초기화하지 않고 TRAIN에서 학습된 불확실성(방패)을 그대로 가져옵니다!
    global current_params, current_state, momentum, thermostat, saved_model_result, param_samples
    
    perf_acc, perf_prec, perf_rec, perf_f1 = {}, {}, {}, {}
    perf_mse, perf_rmse, perf_mae = {}, {}, {}
    cnt_anomalies, actual_anomalies, tp_anomalies = {}, {}, {}
    
    online_ds = tfio.experimental.streaming.KafkaBatchIODataset(
        topics=[TEST_TOPIC],
        group_id=f'cg_test_{method}_{int(time.time())}',
        servers='127.0.0.1:9092',
        stream_timeout=10000,
        configuration=['session.timeout.ms=7000',
                       'max.poll.interval.ms=8000',
                       'auto.offset.reset=earliest'],
    )
    
    iterations_count = math.ceil(len(x_test) / batch_size) - 1
    current_message_count = 0
    
    print(f'\n🧪 TEST 시작 - {method.upper()}')
    test_start = time.time()
    
    for mini_ds in online_ds:
        mini_ds = mini_ds.map(decode_kafka_online_item)
        mini_ds = mini_ds.batch(batch_size)
        
        for (message, key) in mini_ds:
            if current_message_count >= iterations_count:
                break
            current_message_count += 1
            
            msg_np = message.numpy()
            key_np = key.numpy()
            
            # 🌟 2. 악명 높았던 model['selection'] ... 3줄 완벽 삭제 완료
            
            # 🌟 3. TRAIN과 동일하게 Online Learning(학습) 먼저 진행하여 최신화
            fit_model(msg_np, current_message_count,
                      method=method, learning_rate=learning_rate)
            
            # 🌟 4. 최신 파라미터와 방패(bma_uncertainty)를 얹어서 예측 및 평가!
            evaluate_forecast(saved_model_result, msg_np, key_np, current_message_count,
                              batch_conf_interval,
                              perf_acc, perf_prec, perf_rec, perf_f1,
                              perf_mse, perf_rmse, perf_mae,
                              cnt_anomalies, actual_anomalies, tp_anomalies, method=method)
        
        if current_message_count >= iterations_count:
            break
    
    elapsed = time.time() - test_start
    
    return print_results(perf_acc, perf_prec, perf_rec, perf_f1,
                         perf_mse, perf_rmse, perf_mae,
                         cnt_anomalies, actual_anomalies, tp_anomalies,
                         phase='TEST', method=method, elapsed=elapsed)

---
## 10. 실행: MLE (SARIMA 베이스라인)
기존 `Code_StreamingAnomalyDetetctionUsingBayesianARIMA.ipynb`와 동일

In [ ]:
train_result_mle = run_train('mle', learning_rate=1e-5, batch_conf_interval=1.8)

2026-03-03 09:53:07.839568: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: session.timeout.ms=7000
2026-03-03 09:53:07.839588: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: max.poll.interval.ms=8000
2026-03-03 09:53:07.839595: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: auto.offset.reset=earliest
2026-03-03 09:53:07.839600: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: group.id=cg_train_mle_1772499187
2026-03-03 09:53:07.839604: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: bootstrap.servers=127.0.0.1:9092
2026-03-03 09:53:07.839615: I tensorflow_io/core/kernels/kafka_kernels.cc:919] max num of messages per batch: 10000
2026-03-03 09:53:07.839620: I tensorflow_io/core/kernels/kafka_kernels.cc:938] Creating the kafka consumer
2026-03-03 09:53:07.839927: I tensorflow_io/core/kernels/kafka_kernels.cc:945] Subscribing to the kafka topic: train-1772499137
2026-0


🚀 TRAIN 시작 - MLE, lr=1e-05


2026-03-03 09:53:08.483947: I tensorflow_io/core/kernels/kafka_kernels.cc:996] EOF reached for all 1 partition(s)


  Batch 50/378
  Batch 100/378
  Batch 150/378
  Batch 200/378
  Batch 250/378
  Batch 300/378
  Batch 350/378

📊 TRAIN 결과 (MLE)
총 배치      : 378
알람 배치    : 320
실제 이상 배치: 3
TP 배치      : 3
MSE Mean     : 0.00827535
RMSE Mean    : 0.052113
MAE Mean     : 0.052108
Accuracy     : 16.14%
Precision    : 0.94%
Recall       : 100.00%
F1 Score     : 1.86%
Time         : 15.7s
TP Batches   : [51, 84, 342]


2026-03-03 09:53:23.544276: E tensorflow_io/core/kernels/kafka_kernels.cc:1001] Local: Timed out


: 

In [31]:
write_to_kafka(TEST_TOPIC, list(zip(x_test, y_test)))
time.sleep(2)
test_result_mle = run_test('mle', learning_rate=1e-5, batch_conf_interval=1.75)

✅ test-1772496458: 2064개 전송 완료


2026-03-03 09:20:47.879644: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: session.timeout.ms=7000
2026-03-03 09:20:47.879701: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: max.poll.interval.ms=8000
2026-03-03 09:20:47.879717: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: auto.offset.reset=earliest
2026-03-03 09:20:47.879729: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: group.id=cg_test_mle_1772497247
2026-03-03 09:20:47.879740: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: bootstrap.servers=127.0.0.1:9092
2026-03-03 09:20:47.879759: I tensorflow_io/core/kernels/kafka_kernels.cc:919] max num of messages per batch: 10000
2026-03-03 09:20:47.879770: I tensorflow_io/core/kernels/kafka_kernels.cc:938] Creating the kafka consumer
2026-03-03 09:20:47.880056: I tensorflow_io/core/kernels/kafka_kernels.cc:945] Subscribing to the kafka topic: test-1772496458
2026-03-


🧪 TEST 시작 - MLE


2026-03-03 09:20:48.517284: I tensorflow_io/core/kernels/kafka_kernels.cc:996] EOF reached for all 1 partition(s)



📊 TEST 결과 (MLE)
총 배치      : 42
알람 배치    : 37
실제 이상 배치: 3
TP 배치      : 3
MSE Mean     : 0.03678263
RMSE Mean    : 0.145077
MAE Mean     : 0.145077
Accuracy     : 19.05%
Precision    : 8.11%
Recall       : 100.00%
F1 Score     : 15.00%
Time         : 10.6s
TP Batches   : [6, 13, 39]


2026-03-03 09:20:58.519138: E tensorflow_io/core/kernels/kafka_kernels.cc:1001] Local: Timed out


## 11. 실행: GD

In [28]:
train_result_gd = run_train('gd', learning_rate=5e-5, batch_conf_interval=1.97)

2026-03-03 09:16:21.316002: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: session.timeout.ms=7000
2026-03-03 09:16:21.316030: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: max.poll.interval.ms=8000
2026-03-03 09:16:21.316038: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: auto.offset.reset=earliest
2026-03-03 09:16:21.316043: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: group.id=cg_train_gd_1772496981
2026-03-03 09:16:21.316048: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: bootstrap.servers=127.0.0.1:9092
2026-03-03 09:16:21.316060: I tensorflow_io/core/kernels/kafka_kernels.cc:919] max num of messages per batch: 10000
2026-03-03 09:16:21.316065: I tensorflow_io/core/kernels/kafka_kernels.cc:938] Creating the kafka consumer
2026-03-03 09:16:21.316271: I tensorflow_io/core/kernels/kafka_kernels.cc:945] Subscribing to the kafka topic: train-1772496458
2026-03


🚀 TRAIN 시작 - GD, lr=5e-05


2026-03-03 09:16:21.958124: I tensorflow_io/core/kernels/kafka_kernels.cc:996] EOF reached for all 1 partition(s)


  Batch 50/171


2026-03-03 09:16:31.960466: E tensorflow_io/core/kernels/kafka_kernels.cc:1001] Local: Timed out


  Batch 100/171
  Batch 150/171

📊 TRAIN 결과 (GD)
총 배치      : 171
알람 배치    : 63
실제 이상 배치: 2
TP 배치      : 2
MSE Mean     : 0.08544229
RMSE Mean    : 0.269320
MAE Mean     : 0.269320
Accuracy     : 64.33%
Precision    : 3.17%
Recall       : 100.00%
F1 Score     : 6.15%
Time         : 28.3s
TP Batches   : [124, 150]


In [29]:
test_result_gd = run_test('gd', learning_rate=1e-5, batch_conf_interval=1.76)


✅ test-1772496458: 2064개 전송 완료


2026-03-03 09:16:51.825579: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: session.timeout.ms=7000
2026-03-03 09:16:51.825617: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: max.poll.interval.ms=8000
2026-03-03 09:16:51.825625: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: auto.offset.reset=earliest
2026-03-03 09:16:51.825630: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: group.id=cg_test_gd_1772497011
2026-03-03 09:16:51.825635: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: bootstrap.servers=127.0.0.1:9092
2026-03-03 09:16:51.825647: I tensorflow_io/core/kernels/kafka_kernels.cc:919] max num of messages per batch: 10000
2026-03-03 09:16:51.825652: I tensorflow_io/core/kernels/kafka_kernels.cc:938] Creating the kafka consumer
2026-03-03 09:16:51.825946: I tensorflow_io/core/kernels/kafka_kernels.cc:945] Subscribing to the kafka topic: test-1772496458
2026-03-0


🧪 TEST 시작 - GD


2026-03-03 09:16:52.464178: I tensorflow_io/core/kernels/kafka_kernels.cc:996] EOF reached for all 1 partition(s)



📊 TEST 결과 (GD)
총 배치      : 42
알람 배치    : 36
실제 이상 배치: 3
TP 배치      : 3
MSE Mean     : 0.06532983
RMSE Mean    : 0.224017
MAE Mean     : 0.224017
Accuracy     : 21.43%
Precision    : 8.33%
Recall       : 100.00%
F1 Score     : 15.38%
Time         : 10.6s
TP Batches   : [6, 13, 39]


2026-03-03 09:17:02.465483: E tensorflow_io/core/kernels/kafka_kernels.cc:1001] Local: Timed out


## 12. 실행: SGD

In [27]:
train_result_sgd = run_train('sgd', learning_rate=0.001, batch_conf_interval=1.97)

2026-03-03 09:15:05.649178: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: session.timeout.ms=7000
2026-03-03 09:15:05.649199: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: max.poll.interval.ms=8000
2026-03-03 09:15:05.649209: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: auto.offset.reset=earliest
2026-03-03 09:15:05.649216: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: group.id=cg_train_sgd_1772496905
2026-03-03 09:15:05.649222: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: bootstrap.servers=127.0.0.1:9092
2026-03-03 09:15:05.649236: I tensorflow_io/core/kernels/kafka_kernels.cc:919] max num of messages per batch: 10000
2026-03-03 09:15:05.649242: I tensorflow_io/core/kernels/kafka_kernels.cc:938] Creating the kafka consumer
2026-03-03 09:15:05.649404: I tensorflow_io/core/kernels/kafka_kernels.cc:945] Subscribing to the kafka topic: train-1772496458
2026-0


🚀 TRAIN 시작 - SGD, lr=0.001


2026-03-03 09:15:06.286701: I tensorflow_io/core/kernels/kafka_kernels.cc:996] EOF reached for all 1 partition(s)


  Batch 50/171


2026-03-03 09:15:16.288470: E tensorflow_io/core/kernels/kafka_kernels.cc:1001] Local: Timed out


  Batch 100/171
  Batch 150/171
param_samples std: [0.        0.        0.0008639]

📊 TRAIN 결과 (SGD)
총 배치      : 171
알람 배치    : 63
실제 이상 배치: 2
TP 배치      : 2
MSE Mean     : 0.08544229
RMSE Mean    : 0.269320
MAE Mean     : 0.269320
Accuracy     : 64.33%
Precision    : 3.17%
Recall       : 100.00%
F1 Score     : 6.15%
Time         : 28.1s
TP Batches   : [124, 150]


In [26]:
test_result_sgd = run_test('sgd', learning_rate=0.001, batch_conf_interval=1.75)


🧪 TEST 시작 - SGD


2026-03-03 09:14:53.178345: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: session.timeout.ms=7000
2026-03-03 09:14:53.178370: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: max.poll.interval.ms=8000
2026-03-03 09:14:53.178379: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: auto.offset.reset=earliest
2026-03-03 09:14:53.178385: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: group.id=cg_test_sgd_1772496893
2026-03-03 09:14:53.178390: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: bootstrap.servers=127.0.0.1:9092
2026-03-03 09:14:53.178402: I tensorflow_io/core/kernels/kafka_kernels.cc:919] max num of messages per batch: 10000
2026-03-03 09:14:53.178407: I tensorflow_io/core/kernels/kafka_kernels.cc:938] Creating the kafka consumer
2026-03-03 09:14:53.178595: I tensorflow_io/core/kernels/kafka_kernels.cc:945] Subscribing to the kafka topic: test-1772496458
2026-03-


📊 TEST 결과 (SGD)
총 배치      : 42
알람 배치    : 37
실제 이상 배치: 3
TP 배치      : 3
MSE Mean     : 0.06532983
RMSE Mean    : 0.224017
MAE Mean     : 0.224017
Accuracy     : 19.05%
Precision    : 8.11%
Recall       : 100.00%
F1 Score     : 15.00%
Time         : 10.6s
TP Batches   : [6, 13, 39]


2026-03-03 09:15:03.816027: E tensorflow_io/core/kernels/kafka_kernels.cc:1001] Local: Timed out


## 13. 실행: SGLD

In [17]:
run_train('sgld', learning_rate=1e-5, batch_conf_interval=1.97)
#run_test('sgld', learning_rate=1e-5, batch_conf_interval=1.8)

2026-03-03 09:10:37.933367: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: session.timeout.ms=7000
2026-03-03 09:10:37.933396: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: max.poll.interval.ms=8000
2026-03-03 09:10:37.933404: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: auto.offset.reset=earliest
2026-03-03 09:10:37.933409: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: group.id=cg_train_sgld_1772496637
2026-03-03 09:10:37.933415: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: bootstrap.servers=127.0.0.1:9092
2026-03-03 09:10:37.933426: I tensorflow_io/core/kernels/kafka_kernels.cc:919] max num of messages per batch: 10000
2026-03-03 09:10:37.933431: I tensorflow_io/core/kernels/kafka_kernels.cc:938] Creating the kafka consumer
2026-03-03 09:10:37.933654: I tensorflow_io/core/kernels/kafka_kernels.cc:945] Subscribing to the kafka topic: train-1772496458
2026-


🚀 TRAIN 시작 - SGLD, lr=1e-05


2026-03-03 09:10:38.573724: I tensorflow_io/core/kernels/kafka_kernels.cc:996] EOF reached for all 1 partition(s)


  Batch 50/171


2026-03-03 09:10:48.576471: E tensorflow_io/core/kernels/kafka_kernels.cc:1001] Local: Timed out


  Batch 100/171
  Batch 150/171
param_samples std: [0.01212155 0.01907424 0.0132451 ]

📊 TRAIN 결과 (SGLD)
총 배치      : 171
알람 배치    : 59
실제 이상 배치: 2
TP 배치      : 2
MSE Mean     : 0.08544229
RMSE Mean    : 0.269320
MAE Mean     : 0.269320
Accuracy     : 66.67%
Precision    : 3.39%
Recall       : 100.00%
F1 Score     : 6.56%
Time         : 27.1s
TP Batches   : [124, 150]


{'recall': 100.0,
 'accuracy': 66.67,
 'precision': 3.39,
 'f1': 6.56,
 'time': 27.1}

In [24]:
run_test('sgld', learning_rate=1e-5, batch_conf_interval=1.76)

2026-03-03 09:13:13.529079: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: session.timeout.ms=7000
2026-03-03 09:13:13.529112: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: max.poll.interval.ms=8000
2026-03-03 09:13:13.529121: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: auto.offset.reset=earliest
2026-03-03 09:13:13.529127: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: group.id=cg_test_sgld_1772496793
2026-03-03 09:13:13.529132: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: bootstrap.servers=127.0.0.1:9092
2026-03-03 09:13:13.529144: I tensorflow_io/core/kernels/kafka_kernels.cc:919] max num of messages per batch: 10000
2026-03-03 09:13:13.529149: I tensorflow_io/core/kernels/kafka_kernels.cc:938] Creating the kafka consumer
2026-03-03 09:13:13.529381: I tensorflow_io/core/kernels/kafka_kernels.cc:945] Subscribing to the kafka topic: test-1772496458
2026-03


🧪 TEST 시작 - SGLD


2026-03-03 09:13:14.168351: I tensorflow_io/core/kernels/kafka_kernels.cc:996] EOF reached for all 1 partition(s)
2026-03-03 09:13:24.169860: E tensorflow_io/core/kernels/kafka_kernels.cc:1001] Local: Timed out



📊 TEST 결과 (SGLD)
총 배치      : 42
알람 배치    : 32
실제 이상 배치: 3
TP 배치      : 2
MSE Mean     : 0.06532983
RMSE Mean    : 0.224017
MAE Mean     : 0.224017
Accuracy     : 26.19%
Precision    : 6.25%
Recall       : 66.67%
F1 Score     : 11.43%
Time         : 10.9s
TP Batches   : [13, 39]


{'recall': 66.67,
 'accuracy': 26.19,
 'precision': 6.25,
 'f1': 11.43,
 'time': 10.9}

## 14. 실행: SGHMC

In [32]:
train_result_sghmc = run_train('sghmc', learning_rate=1e-5, batch_conf_interval=1.97)

2026-03-03 09:22:50.293998: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: session.timeout.ms=7000
2026-03-03 09:22:50.294030: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: max.poll.interval.ms=8000
2026-03-03 09:22:50.294039: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: auto.offset.reset=earliest
2026-03-03 09:22:50.294044: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: group.id=cg_train_sghmc_1772497370
2026-03-03 09:22:50.294050: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: bootstrap.servers=127.0.0.1:9092
2026-03-03 09:22:50.294061: I tensorflow_io/core/kernels/kafka_kernels.cc:919] max num of messages per batch: 10000
2026-03-03 09:22:50.294066: I tensorflow_io/core/kernels/kafka_kernels.cc:938] Creating the kafka consumer
2026-03-03 09:22:50.294328: I tensorflow_io/core/kernels/kafka_kernels.cc:945] Subscribing to the kafka topic: train-1772496458
2026


🚀 TRAIN 시작 - SGHMC, lr=1e-05


2026-03-03 09:22:50.933349: I tensorflow_io/core/kernels/kafka_kernels.cc:996] EOF reached for all 1 partition(s)


  Batch 50/171


2026-03-03 09:23:00.934789: E tensorflow_io/core/kernels/kafka_kernels.cc:1001] Local: Timed out


  Batch 100/171
  Batch 150/171
param_samples std: [0.00770484 0.01003358 0.13657414]

📊 TRAIN 결과 (SGHMC)
총 배치      : 171
알람 배치    : 45
실제 이상 배치: 2
TP 배치      : 1
MSE Mean     : 0.08544229
RMSE Mean    : 0.269320
MAE Mean     : 0.269320
Accuracy     : 73.68%
Precision    : 2.22%
Recall       : 50.00%
F1 Score     : 4.26%
Time         : 41.6s
TP Batches   : [124]


In [33]:
test_result_sghmc = run_test('sghmc', learning_rate=0.001, batch_conf_interval=1.75)


🧪 TEST 시작 - SGHMC


2026-03-03 09:23:31.902223: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: session.timeout.ms=7000
2026-03-03 09:23:31.902248: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: max.poll.interval.ms=8000
2026-03-03 09:23:31.902257: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: auto.offset.reset=earliest
2026-03-03 09:23:31.902263: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: group.id=cg_test_sghmc_1772497411
2026-03-03 09:23:31.902268: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: bootstrap.servers=127.0.0.1:9092
2026-03-03 09:23:31.902279: I tensorflow_io/core/kernels/kafka_kernels.cc:919] max num of messages per batch: 10000
2026-03-03 09:23:31.902284: I tensorflow_io/core/kernels/kafka_kernels.cc:938] Creating the kafka consumer
2026-03-03 09:23:31.923584: I tensorflow_io/core/kernels/kafka_kernels.cc:945] Subscribing to the kafka topic: test-1772496458
2026-0


📊 TEST 결과 (SGHMC)
총 배치      : 42
알람 배치    : 8
실제 이상 배치: 3
TP 배치      : 1
MSE Mean     : 0.06532983
RMSE Mean    : 0.224017
MAE Mean     : 0.224017
Accuracy     : 78.57%
Precision    : 12.50%
Recall       : 33.33%
F1 Score     : 18.18%
Time         : 10.6s
TP Batches   : [13]


2026-03-03 09:23:42.565996: E tensorflow_io/core/kernels/kafka_kernels.cc:1001] Local: Timed out


## 15. 실행: SGNHT

In [ ]:
train_result_sgnht = run_train('sgnht', learning_rate=0.001, batch_conf_interval=1.97)

2026-03-03 09:26:50.311147: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: session.timeout.ms=7000
2026-03-03 09:26:50.311175: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: max.poll.interval.ms=8000
2026-03-03 09:26:50.311184: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: auto.offset.reset=earliest
2026-03-03 09:26:50.311189: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: group.id=cg_train_sgnht_1772497610
2026-03-03 09:26:50.311195: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: bootstrap.servers=127.0.0.1:9092
2026-03-03 09:26:50.311206: I tensorflow_io/core/kernels/kafka_kernels.cc:919] max num of messages per batch: 10000
2026-03-03 09:26:50.311212: I tensorflow_io/core/kernels/kafka_kernels.cc:938] Creating the kafka consumer
2026-03-03 09:26:50.311394: I tensorflow_io/core/kernels/kafka_kernels.cc:945] Subscribing to the kafka topic: train-1772496458
2026


🚀 TRAIN 시작 - SGNHT, lr=1e-05


2026-03-03 09:26:50.952125: I tensorflow_io/core/kernels/kafka_kernels.cc:996] EOF reached for all 1 partition(s)


  Batch 50/171


2026-03-03 09:27:00.954464: E tensorflow_io/core/kernels/kafka_kernels.cc:1001] Local: Timed out


  Batch 100/171
  Batch 150/171
param_samples std: [0.00938823 0.00791644 0.01084153]

📊 TRAIN 결과 (SGNHT)
총 배치      : 171
알람 배치    : 60
실제 이상 배치: 2
TP 배치      : 2
MSE Mean     : 0.08544229
RMSE Mean    : 0.269320
MAE Mean     : 0.269320
Accuracy     : 66.08%
Precision    : 3.33%
Recall       : 100.00%
F1 Score     : 6.45%
Time         : 26.3s
TP Batches   : [124, 150]


In [37]:
test_result_sgnht = run_test('sgnht', learning_rate=0.001, batch_conf_interval=1.75)

2026-03-03 09:34:25.266867: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: session.timeout.ms=7000
2026-03-03 09:34:25.266894: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: max.poll.interval.ms=8000
2026-03-03 09:34:25.266901: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: auto.offset.reset=earliest
2026-03-03 09:34:25.266907: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: group.id=cg_test_sgnht_1772498065
2026-03-03 09:34:25.266911: I tensorflow_io/core/kernels/kafka_kernels.cc:879] Kafka configuration: bootstrap.servers=127.0.0.1:9092
2026-03-03 09:34:25.266921: I tensorflow_io/core/kernels/kafka_kernels.cc:919] max num of messages per batch: 10000
2026-03-03 09:34:25.266926: I tensorflow_io/core/kernels/kafka_kernels.cc:938] Creating the kafka consumer
2026-03-03 09:34:25.267540: I tensorflow_io/core/kernels/kafka_kernels.cc:945] Subscribing to the kafka topic: test-1772496458
2026-0


🧪 TEST 시작 - SGNHT


2026-03-03 09:34:25.904910: I tensorflow_io/core/kernels/kafka_kernels.cc:996] EOF reached for all 1 partition(s)



📊 TEST 결과 (SGNHT)
총 배치      : 42
알람 배치    : 36
실제 이상 배치: 3
TP 배치      : 3
MSE Mean     : 0.06532983
RMSE Mean    : 0.224017
MAE Mean     : 0.224017
Accuracy     : 21.43%
Precision    : 8.33%
Recall       : 100.00%
F1 Score     : 15.38%
Time         : 10.6s
TP Batches   : [6, 13, 39]


2026-03-03 09:34:35.906942: E tensorflow_io/core/kernels/kafka_kernels.cc:1001] Local: Timed out


## 16. 전체 결과 테이블

In [ ]:
all_train = {
    'MLE': train_result_mle,
    'GD': train_result_gd,
    'SGD': train_result_sgd,
    'SGLD': train_result_sgld,
    'SGHMC': train_result_sghmc,
    'SGNHT': train_result_sgnht,
}

all_test = {
    'MLE': test_result_mle,
    'GD': test_result_gd,
    'SGD': test_result_sgd,
    'SGLD': test_result_sgld,
    'SGHMC': test_result_sghmc,
    'SGNHT': test_result_sgnht,
}

for phase, results in [('TRAIN', all_train), ('TEST', all_test)]:
    print(f'\n{"="*80}')
    print(f'  {phase} Results - {DATASET}')
    print(f'{"="*80}')
    print(f'{"Method":<10} {"Recall(%)":<12} {"Accuracy(%)":<14} {"Precision(%)":<14} {"F1(%)":<10} {"Time(s)":<10}')
    print(f'{"-"*80}')
    for m, r in results.items():
        print(f'{m:<10} {r["recall"]:<12} {r["accuracy"]:<14} {r["precision"]:<14} {r["f1"]:<10} {r["time"]:<10}')

## 17. Kafka 종료

In [ ]:
!pkill -f kafka 2>/dev/null || true
!pkill -f zookeeper 2>/dev/null || true
print('✅ Kafka & Zookeeper 종료 완료')